# WC-MNIST Parameter-Exploration — Clean Redo (500 × 3 configs)

Runs the no-training parameter-exploration study: for each of 500 seeds and 3
configs (`simultaneous`, `sequential_noH`, `sequential_H`), a single **10-second
free-running simulation** of a freshly-initialised random Wilson-Cowan network
driven by MNIST — **no training** (no gradient steps, no epoch loop).

Each run saves a reproducibility bundle to Google Drive:
`arrays.npz` (E/I trajectories) + `params.json` + `manifest.json`, plus optional
figures. The sweep is **resumable** — re-run this notebook after a disconnect and
it skips completed runs.

**Runtime**: ~3–4 s/run on GPU → 1500 runs ≈ 1.5–2 h. CPU is ~1.5× slower.
**Storage**: compressed f16 arrays ≈ 5 MB/run → 1500 runs ≈ 7–8 GB on Drive.


## 1 · Mount Drive & choose output location

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All bundles + master log land here. Safe to change; must be on Drive to persist.
OUT_ROOT = '/content/drive/MyDrive/wc_mnist_param_explore'
import os
os.makedirs(OUT_ROOT, exist_ok=True)
print('Output root:', OUT_ROOT)


## 2 · Clone the pinned code

The study is pinned to a tagged commit so every run's `manifest.json` records the exact code version.

In [ ]:
REPO   = 'https://github.com/AlexBoro/wc-mnist-study.git'
TAG    = 'param-explore-v1'          # <-- pinned release tag
CODE_DIR = '/content/wc-mnist-study'

import os, subprocess
if not os.path.isdir(CODE_DIR):
    subprocess.run(['git','clone','--branch',TAG,'--depth','1',REPO,CODE_DIR], check=True)
else:
    print('code dir exists; leaving as-is (delete to re-clone a different tag)')

# Record the exact commit so manifests carry it even without a .git in the run cwd
commit = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
os.environ['WC_CODE_COMMIT'] = commit
print('Pinned commit:', commit)

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


## 3 · Dependencies

Colab ships torch + torchvision + numpy + scipy. Install only what's missing.

In [ ]:
import importlib, subprocess, sys
for pkg in ['torch','torchvision','numpy','scipy','matplotlib']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))


## 4 · Sweep configuration

In [ ]:
import numpy as np
os.chdir(CODE_DIR)                       # so wc_experiment resolves MNIST + git

import wc_config as cfg
import param_explore as PE
import param_persist as PP
import param_render as PR

# --- knobs -------------------------------------------------------------------
N_SEEDS       = 500                      # 500 seeds x 3 configs = 1500 runs
SEEDS         = list(range(N_SEEDS))
CONFIG_NAMES  = [c.name for c in cfg.make_configs()]   # simultaneous, sequential_noH, sequential_H
RUN_SECONDS   = 10.0
RECORD_STRIDE = 1                        # 1 = full dt resolution (5000 steps); 2 halves storage
DTYPE_OUT     = np.float16               # f16 compressed ~5 MB/run
RENDER_FIGS   = True                     # write heatmap_E.png + lfp.png per run
# -----------------------------------------------------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
render_fn = (lambda out, res: PR.render_from_result(out, res)) if RENDER_FIGS else None
print(f'{len(CONFIG_NAMES)} configs x {N_SEEDS} seeds = {len(CONFIG_NAMES)*N_SEEDS} runs',
      f'| device={device} | stride={RECORD_STRIDE} | dtype={np.dtype(DTYPE_OUT).name}')


## 5 · MNIST download (once)

Downloads MNIST into the repo's data root; subsequent runs reuse it.

In [ ]:
from torchvision import datasets, transforms
import wc_experiment as X
data_root = X._resolve_data_root()
datasets.MNIST(root=data_root, download=True, train=True, transform=transforms.ToTensor())
print('MNIST ready at', data_root)


## 6 · Run the sweep (resumable)

Safe to interrupt and re-run: completed bundles are skipped. A disconnect costs at most the in-flight run.

In [ ]:
result = PP.run_sweep(
    OUT_ROOT, seeds=SEEDS, config_names=CONFIG_NAMES, device=device,
    run_seconds=RUN_SECONDS, record_stride=RECORD_STRIDE, dtype_out=DTYPE_OUT,
    code_tag=TAG, repo_dir=CODE_DIR, render_fn=render_fn, verbose=True)
print(result)


## 7 · Verify completeness

In [ ]:
import glob, json
n_expected = len(CONFIG_NAMES) * len(SEEDS)
complete = 0
missing = []
for cname in CONFIG_NAMES:
    for seed in SEEDS:
        if PP.is_complete(OUT_ROOT, cname, seed):
            complete += 1
        else:
            missing.append((cname, seed))
print(f'complete: {complete}/{n_expected}')
if missing:
    print('missing (re-run cell 6 to fill):', missing[:20], '...' if len(missing)>20 else '')

log_path = os.path.join(OUT_ROOT, PP.MASTER_LOG)
if os.path.exists(log_path):
    import csv
    rows = list(csv.DictReader(open(log_path)))
    print(f'master log rows: {len(rows)}')


## 8 · (optional) Zip a subset for download

The full 7–8 GB stays on Drive. To pull a handful of bundles locally for
inspection, zip the run dirs you want.

In [ ]:
# import shutil
# subset = [('sequential_H', 0), ('simultaneous', 0), ('sequential_noH', 0)]
# stage = '/content/pe_subset'; os.makedirs(stage, exist_ok=True)
# for cname, seed in subset:
#     src = PP.run_dir(OUT_ROOT, cname, seed)
#     shutil.copytree(src, os.path.join(stage, f'{cname}_run_{seed:03d}'), dirs_exist_ok=True)
# shutil.make_archive('/content/pe_subset', 'zip', stage)
# from google.colab import files; files.download('/content/pe_subset.zip')
